# Análise Exploratória

In [1]:
import sys
import numpy as np
import seaborn as sns
import pandas as pd
np.set_printoptions(threshold=sys.maxsize)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 80)
def id_like_dtypes(path):
    """Detecta colunas de identificação/código, CPF e CNPJ pelo nome e monta
    um dict de dtype=str para evitar perda de zeros à esquerda na leitura."""
    header = pd.read_csv(path, nrows=0).columns
    dtypes = {}
    for col in header:
        c = col.lower()
        is_id = (
            "codigo" in c or "cnpj" in c or "cpf" in c
            or c.startswith(("id", "numero", "sequencial", "cod"))
            or c.endswith(("id", "idpncp"))
            or c in {"nifornecedor"}
        )
        if is_id:
            dtypes[col] = str
    return dtypes


def read_csv_ids_str(path, dtype=None, **kwargs):
    """pd.read_csv que força colunas de identificação/código/CPF/CNPJ como string."""
    dtypes = id_like_dtypes(path)
    if dtype:
        dtypes.update(dtype)
    return pd.read_csv(path, dtype=dtypes, **kwargs)

In [2]:
raw_arp = read_csv_ids_str('raw_arp_202607071230.csv')
raw_arp_item = read_csv_ids_str('raw_arp_item_202607071231.csv', low_memory=False)
raw_arp_unidades_item = read_csv_ids_str('raw_arp_unidades_item_202607071231.csv', low_memory=False)
raw_arp_adesoes_item = read_csv_ids_str('raw_arp_adesoes_item_202607071231.csv', low_memory=False)
raw_arp_empenhos_saldo = read_csv_ids_str('raw_arp_empenhos_saldo_202607071231.csv')

# Domínio 6: ARP — Atas de Registro de Preços

Atas resultantes de licitações do tipo SRP (Sistema de Registro de Preços).

## `raw_arp`
Cabeçalho da ata de registro de preço.

| Coluna | Descrição |
|---|---|
| `numeroataregistropreco` **PK** | Número da ata |
| `idcompra` **PK/FK** → `raw_contratacoes` | Processo de contratação de origem |
| `codigoorgao` **FK** → `raw_orgao` | Órgão gerenciador |
| `codigounidadegerenciadora` | Código da UASG gerenciadora *(pode ser nulo — ver nota)* |
| `valortotal` | Valor total da ata |
| `datavigenciainicial` | Início da vigência |
| `datavigenciafinal` | Fim da vigência |
| `statusata` | Vigente, encerrada, etc. |
| `objeto` | Objeto da ata |
| `quantidadeitens` | Número de itens registrados |
| `ataexcluido` | Flag de exclusão lógica |

> **Nota:** `codigounidadegerenciadora` pode ser nulo. A chave composta usa `idcompra` como alternativa.

In [3]:
raw_arp.shape

(65483, 26)

In [4]:
raw_arp.duplicated().sum()

0

In [5]:
raw_arp.head(3)

,numeroataregistropreco,codigounidadegerenciadora,nomeunidadegerenciadora,codigoorgao,nomeorgao,linkatapncp,linkcomprapncp,numerocompra,anocompra,codigomodalidadecompra,nomemodalidadecompra,dataassinatura,datavigenciainicial,datavigenciafinal,valortotal,statusata,objeto,quantidadeitens,datahoraatualizacao,datahorainclusao,datahoraexclusao,ataexcluido,numerocontrolepncpata,numerocontrolepncpcompra,idcompra,dt_ingest
0,00002/2026,080102,ESP-COORDENADORIA GERAL DE SUPORTE ADMINISTRATIVO,95561.0,ESP-SECRETARIA DA EDUCACAO,https://pncp.gov.br/app/atas/46384111000140/2024/1469/1,https://pncp.gov.br/app/editais/46384111000140/2024/001469,90040,2024,05,Pregão,2026-01-30,2026-01-31,2027-01-30,0.00,Ata de Registro de Preços,aquisição de passagens aereas,1.0,2026-02-18T19:00:27,2025-01-30T10:42:49,NaN,False,46384111000140-1-001469/2024-000001,46384111000140-1-001469/2024,08010205900402024,2026-06-30T19:25:53.143259
1,00001/2025,200340,DIRETORIA DE ENSINO DA ANP/PF,30108.0,POLICIA FEDERAL/MJSP,https://pncp.gov.br/app/atas/00394494000136/2024/552/20,https://pncp.gov.br/app/editais/00394494000136/2024/000552,90002,2024,05,Pregão,2025-01-31,2026-02-04,2027-02-04,2239105.18,Ata de Registro de Preços,Registro de preços para a eventual aquisição de materiais e equipamentos des...,1.0,2026-03-18T14:31:31,2025-02-03T15:19:55,NaN,False,00394494000136-1-000552/2024-000020,00394494000136-1-000552/2024,20034005900022024,2026-06-30T19:25:53.143259
2,00201/2025,925120,SECRETARIA DE ESTADO DA SAUDE/ES,85539.0,SECRETARIA DE ESTADO DA SAÚDE - ES,https://pncp.gov.br/app/atas/27080605000196/2025/44/1,https://pncp.gov.br/app/editais/27080605000196/2025/000044,91001,2025,05,Pregão,2025-02-13,2026-02-14,2027-02-13,276230.40,Ata de Registro de Preços,"DIETA ENTERAL LIQUIDA,POLIMERICA \nDIETA ENTERAL LIQUIDA \nMODULO DE PROTEIN...",16.0,2025-11-25T10:51:21,2025-02-13T16:37:00,NaN,False,27080605000196-1-000044/2025-000001,27080605000196-1-000044/2025,92512005910012025,2026-06-30T19:25:53.143259


### `numeroataregistropreco`
Número da Ata de Registro de Preços

In [6]:
raw_arp.numeroataregistropreco.value_counts(1, dropna=False).head()

numeroataregistropreco
00001/2026    0.018188
00002/2026    0.015653
00003/2026    0.014981
00004/2026    0.014034
00005/2026    0.013179
Name: proportion, dtype: float64

#### Disperso, ata mais frequente concentra 1,8% dos registros.

### `codigounidadegerenciadora`
Código da unidade gerenciadora

In [7]:
raw_arp.codigounidadegerenciadora.value_counts(1, dropna=False).head()

codigounidadegerenciadora
092501    0.009056
155007    0.008430
925998    0.008109
155125    0.008048
532101    0.007742
Name: proportion, dtype: float64

#### Disperso, UASG mais frequente concentra 0,9% das atas.

### `nomeunidadegerenciadora`
Nome da unidade gerenciadora

In [8]:
raw_arp.nomeunidadegerenciadora.value_counts(1, dropna=False).head()

nomeunidadegerenciadora
ESP-HOSPITAL DAS CLINICAS DE BOTUCATU                 0.009056
EMPRESA BRASILEIRA DE SERVICOS HOSPITALARES           0.008430
AGENCIA DE MODERNIZACAO DA GEST. DE PROCESSOS - AL    0.008109
HOSPITAL UNIVERSITARIO DE SANTA MARIA                 0.008048
ESP-INST. ASSIST. MEDICA SERV. PUB. ESTADUAL          0.007742
Name: proportion, dtype: float64

#### Redundante com `codigounidadegerenciadora` — mesmos percentuais.

### `codigoorgao`
Código do órgão

In [9]:
raw_arp.codigoorgao.value_counts(1, dropna=False).head()

codigoorgao
52121      0.239329
26443      0.133103
52111      0.056045
52131      0.048455
52121.0    0.027641
Name: proportion, dtype: float64

#### (TO DO) O mesmo código de órgão aparece em formatos diferentes — `52121` (23,9%) e `52121.0` (2,8%) — provável inconsistência de formatação na fonte, mesmo com a coluna forçada como string na leitura.

### `nomeorgao`
Nome do órgão

In [10]:
raw_arp.nomeorgao.value_counts(1, dropna=False).head()

nomeorgao
COMANDO DO EXERCITO                            0.266970
EMPRESA BRASILEIRA DE SERVICOS HOSPITALARES    0.149138
COMANDO DA AERONAUTICA                         0.061833
COMANDO DA MARINHA                             0.054839
ESTADO DO RIO DE JANEIRO                       0.029855
Name: proportion, dtype: float64

#### 27% Comando do Exército, 15% EBSERH, 6% Comando da Aeronáutica.

### `linkatapncp`
Link para acesso à ata no PNCP

In [11]:
raw_arp.linkatapncp.value_counts(0, dropna=False).head()

linkatapncp
NaN                                                         1990
https://pncp.gov.br/app/atas/00394502000144/2025/12183/1       2
https://pncp.gov.br/app/atas/00394452000103/2025/20564/9       2
https://pncp.gov.br/app/atas/07424905000138/2025/314/2         2
https://pncp.gov.br/app/atas/68576800000199/2025/40/1          2
Name: count, dtype: int64

#### 97% dos valores nulos (cerca de 1990 de ~65 mil atas sem o link).

### `linkcomprapncp`
Link para acesso à compra no PNCP

In [12]:
raw_arp.linkcomprapncp.value_counts(0, dropna=False).head()

linkcomprapncp
NaN                                                           1990
https://pncp.gov.br/app/editais/46384111000140/2026/000204      87
https://pncp.gov.br/app/editais/00394452000103/2025/024882      81
https://pncp.gov.br/app/editais/00394452000103/2026/006634      74
https://pncp.gov.br/app/editais/00394452000103/2025/019660      71
Name: count, dtype: int64

#### 97% dos valores nulos, mesmo padrão de `linkatapncp`.

### `numerocompra`
Número da compra associada à ata

In [13]:
raw_arp.numerocompra.value_counts(1, dropna=False).head()

numerocompra
90001    0.061023
90002    0.049677
90003    0.042927
90005    0.033658
90004    0.029870
Name: proportion, dtype: float64

#### Disperso, compra mais frequente (90001) concentra 6,1% das atas.

### `anocompra`
Ano da compra

In [14]:
raw_arp.anocompra.value_counts(1, dropna=False).head()

anocompra
2025    0.621062
2026    0.368722
2024    0.010079
2027    0.000137
Name: proportion, dtype: float64

#### 62% de 2025, 37% de 2026.

### `codigomodalidadecompra`
Código da modalidade de compra associada à ata

In [15]:
raw_arp.codigomodalidadecompra.value_counts(1, dropna=False).head()

codigomodalidadecompra
05    0.988577
06    0.009926
03    0.000855
07    0.000641
Name: proportion, dtype: float64

#### 99% código 05 (Pregão).

### `nomemodalidadecompra`
Nome da modalidade de compra associada à ata

In [16]:
raw_arp.nomemodalidadecompra.value_counts(1, dropna=False).head()

nomemodalidadecompra
Pregão             0.988577
Dispensa           0.009926
Concorrência       0.000855
Inexigibilidade    0.000641
Name: proportion, dtype: float64

#### Mesma distribuição de `codigomodalidadecompra` — 99% Pregão.

### `dataassinatura`
Data da assinatura

In [17]:
raw_arp.dataassinatura.value_counts(1, dropna=False).head()

dataassinatura
2026-06-08    0.012858
2026-04-28    0.011667
2026-03-26    0.011438
2026-05-20    0.011285
2026-05-04    0.010873
Name: proportion, dtype: float64

#### Sem data dominante, disperso ao longo de 2026.

### `datavigenciainicial`
Data inicial da vigência

In [18]:
raw_arp.datavigenciainicial.value_counts(1, dropna=False).head()

datavigenciainicial
2026-05-07    0.012660
2026-05-21    0.011774
2026-03-31    0.011316
2026-06-03    0.011316
2026-05-22    0.011209
Name: proportion, dtype: float64

#### Sem data dominante, concentrado no primeiro semestre de 2026.

### `datavigenciafinal`
Data final da vigência

In [19]:
raw_arp.datavigenciafinal.value_counts(0, dropna=False).head()

datavigenciafinal
2027-05-07    759
2027-05-21    743
2027-03-26    694
2027-03-18    693
2027-03-31    689
Name: count, dtype: int64

#### Datas concentradas em 2027, cerca de um ano após `datavigenciainicial` — coerente com vigência padrão de 1 ano.

### `valortotal`
Valor total registrado

In [20]:
raw_arp.valortotal.value_counts(1, dropna=False).head()

valortotal
0.0       0.004734
3000.0    0.001176
6000.0    0.001069
9000.0    0.000870
3600.0    0.000870
Name: proportion, dtype: float64

#### 0,47% das atas com valor total igual a zero.

### `statusata`
Status atual da ata (ex.: ativa, encerrada, excluída)

In [21]:
raw_arp.statusata.value_counts(1, dropna=False).head()

statusata
Ata de Registro de Preços    0.995388
Cancelada                    0.004551
Em elaboração                0.000046
NaN                          0.000015
Name: proportion, dtype: float64

#### 99,5% "Ata de Registro de Preços" (vigente), 0,46% Cancelada.

### `objeto`
Objeto ou descrição resumida

In [22]:
raw_arp.objeto.value_counts(0, dropna=False).head()

objeto
REGISTRO DE PREÇO DE MEDICAMENTOS       128
INSUMOS HOSPITALARES                    117
MEDICAMENTOS                            116
Aquisição de material permanente        107
Aquisição de material de expediente.     86
Name: count, dtype: int64

#### Objetos mais frequentes ligados a medicamentos e insumos hospitalares.

### `quantidadeitens`
Quantidade de itens registrados

In [23]:
raw_arp.quantidadeitens.value_counts(1, dropna=False).head()

quantidadeitens
1.0    0.425332
2.0    0.188721
3.0    0.084007
4.0    0.064658
5.0    0.038621
Name: proportion, dtype: float64

#### 42,5% das atas têm apenas 1 item registrado.

### `datahoraatualizacao`
Data e hora da última atualização

In [24]:
raw_arp.datahoraatualizacao.value_counts(1, dropna=False).head()

datahoraatualizacao
2026-01-15T14:15:56    0.000916
2026-01-15T14:15:50    0.000443
2026-01-15T14:15:51    0.000428
2026-01-15T14:15:55    0.000382
2026-01-15T14:15:52    0.000336
Name: proportion, dtype: float64

#### Sem timestamp dominante.

### `datahorainclusao`
Data e hora de inclusão da ata no sistema

In [25]:
raw_arp.datahorainclusao.value_counts(1, dropna=False).head()

datahorainclusao
2026-01-13T15:02:09    0.000046
2026-05-29T15:08:57    0.000046
2026-04-29T14:49:34    0.000046
2026-05-21T14:57:50    0.000046
2026-01-06T10:16:14    0.000031
Name: proportion, dtype: float64

#### Sem timestamp dominante.

### `datahoraexclusao`
Data e hora de exclusão

In [26]:
raw_arp.datahoraexclusao.value_counts(1, dropna=False).head()

datahoraexclusao
NaN    1.0
Name: proportion, dtype: float64

#### 100% dos valores nulos — nenhuma ata excluída na base, coerente com `ataexcluido` 100% False.

### `ataexcluido`
Indicador se a ata foi excluída (0 – False/Não, 1 – True/Sim)

In [27]:
raw_arp.ataexcluido.value_counts(1, dropna=False).head()

ataexcluido
False    1.0
Name: proportion, dtype: float64

#### 100% False — nenhuma ata excluída na base.

### `numerocontrolepncpata`
Número de controle da ata no PNCP

In [28]:
raw_arp.numerocontrolepncpata.value_counts(1, dropna=False).head()

numerocontrolepncpata
NaN                                    0.030390
00394502000144-1-012183/2025-000001    0.000031
00394452000103-1-020564/2025-000009    0.000031
07424905000138-1-000314/2025-000002    0.000031
68576800000199-1-000040/2025-000001    0.000031
Name: proportion, dtype: float64

#### 3% dos valores nulos.

### `numerocontrolepncpcompra`
Número de controle da compra no PNCP

In [29]:
raw_arp.numerocontrolepncpcompra.value_counts(1, dropna=False).head()

numerocontrolepncpcompra
46384111000140-1-000204/2026    0.001390
00394452000103-1-024882/2025    0.001237
00394452000103-1-006634/2026    0.001161
00394452000103-1-019660/2025    0.001115
00394452000103-1-025322/2025    0.001054
Name: proportion, dtype: float64

#### Disperso, sem valor dominante.

### `idcompra`
Identificador único da compra associada

In [30]:
raw_arp.idcompra.value_counts(1, dropna=False).head()

idcompra
08035805900182026    0.001390
16007805900142025    0.001237
16042805900072026    0.001161
16052405900022025    0.001115
16039505900072025    0.001054
Name: proportion, dtype: float64

#### Disperso, mesmo padrão de `numerocontrolepncpcompra`.

## `raw_arp_item`
Itens registrados em cada ata — um item por fornecedor habilitado.

| Coluna | Descrição |
|---|---|
| `numeroataregistropreco` **PK/FK** → `raw_arp` | Número da ata |
| `codigounidadegerenciadora` **PK** | UASG gerenciadora |
| `numeroitem` **PK** | Número do item na ata |
| `idcompra` **PK** | ID da compra de origem |
| `nifornecedor` **PK** | Fornecedor com preço registrado |
| `codigoitem` | Código do item no catálogo |
| `descricaoitem` | Descrição do item |
| `quantidadehomologadaitem` | Quantidade total registrada na ata |
| `valorunitario` | Preço unitário registrado |
| `valortotal` | Valor total registrado |
| `maximoadesao` | Quantidade máxima para adesão (carona) |
| `nomerazaosocialfornecedor` | Nome do fornecedor |
| `itemexcluido` | Flag de exclusão lógica |

> **Nota:** 13.553 registros (~3,5%) têm pelo menos um campo da PK nulo e foram descartados na ingestão.


In [31]:
raw_arp_item.shape

(383772, 35)

In [32]:
raw_arp_item.duplicated().sum()

250

In [33]:
raw_arp_item.head(2)

,numeroataregistropreco,codigounidadegerenciadora,numerocompra,anocompra,codigomodalidadecompra,dataassinatura,datavigenciainicial,datavigenciafinal,numeroitem,codigoitem,descricaoitem,tipoitem,quantidadehomologadaitem,classificacaofornecedor,nifornecedor,nomerazaosocialfornecedor,quantidadehomologadavencedor,valorunitario,valortotal,maximoadesao,nomeunidadegerenciadora,nomemodalidadecompra,idcompra,numerocontrolepncpcompra,datahorainclusao,datahoraatualizacao,quantidadeempenhada,percentualmaiordesconto,situacaosicaf,datahoraexclusao,itemexcluido,numerocontrolepncpata,codigopdm,nomepdm,dt_ingest
0,00012/2026,733100,90003,2025.0,05,2026-02-03T00:00:00,2026-02-04,2027-02-04,00028,480368,"EMBUTIDO, TIPO LINGUIÇA DE PERNIL, TAMANHO GROSSA, TIPO PREPARAÇÃO FRESCA, E...",Material,1232.0,1,01220638000109,WS DISTRIBUIDORA DE ALIMENTOS LTDA,1232,12.4,15276.8,2464.0,CENTRO DE AVALIACAO DA ILHA DA MARAMBAIA,Pregão,73310005900032025,00394502000144-1-008888/2025,2026-02-09T22:57:12,2026-02-09T22:57:12,0.0,0.0,1,NaN,False,00394502000144-1-008888/2025-000017,9806.0,EMBUTIDO,2026-06-30T20:00:21.678702
1,00012/2026,733100,90003,2025.0,05,2026-02-03T00:00:00,2026-02-04,2027-02-04,00035,464592,"LEGUME PROCESSADO, TIPO MANDIOCA, FORMATO PALITO, PREPARO PRÉ FRITO, APRESEN...",Material,1200.0,1,01220638000109,WS DISTRIBUIDORA DE ALIMENTOS LTDA,1200,8.9,10680.0,2400.0,CENTRO DE AVALIACAO DA ILHA DA MARAMBAIA,Pregão,73310005900032025,00394502000144-1-008888/2025,2026-02-09T22:57:12,2026-02-09T22:57:12,0.0,0.0,1,NaN,False,00394502000144-1-008888/2025-000017,19794.0,LEGUME PROCESSADO,2026-06-30T20:00:21.678702


### `numeroataregistropreco`
Número da Ata de Registro de Preços associada ao item

In [34]:
raw_arp_item.numeroataregistropreco.value_counts(1, dropna=False).head()

numeroataregistropreco
00001/2026    0.042270
00002/2026    0.025911
00003/2026    0.023204
00004/2026    0.019707
00005/2026    0.017372
Name: proportion, dtype: float64

#### 4,2% dos itens pertencem à ata 00001/2026 — concentração maior que no cabeçalho da ata (1,8%), coerente com atas maiores terem mais itens.

### `codigounidadegerenciadora`
Código da unidade gerenciadora

In [35]:
raw_arp_item.codigounidadegerenciadora.value_counts(1, dropna=False).head()

codigounidadegerenciadora
120645    0.011830
981083    0.007635
155125    0.006691
160495    0.006509
985811    0.006420
Name: proportion, dtype: float64

#### Disperso, UASG mais frequente concentra 1,2% dos itens.

### `numerocompra`
Número da compra associada ao item

In [36]:
raw_arp_item.numerocompra.value_counts(1, dropna=False).head()

numerocompra
NaN      0.110308
90001    0.068251
90002    0.051979
90003    0.042197
90004    0.030573
Name: proportion, dtype: float64

#### 11% dos valores nulos.

### `anocompra`
Ano da compra

In [37]:
raw_arp_item.anocompra.value_counts(1, dropna=False).head()

anocompra
2025.0    0.532415
2026.0    0.351261
NaN       0.110308
2024.0    0.005985
2027.0    0.000031
Name: proportion, dtype: float64

#### 11% nulos, coerente com `numerocompra`; 53% de 2025, 35% de 2026.

### `codigomodalidadecompra`
Código da modalidade de compra

In [38]:
raw_arp_item.codigomodalidadecompra.value_counts(1, dropna=False).head()

codigomodalidadecompra
05    0.991224
06    0.007379
07    0.001027
03    0.000370
Name: proportion, dtype: float64

#### 99% código 05 (Pregão), mesma distribuição do cabeçalho da ata.

### `dataassinatura`
Data de assinatura

In [39]:
raw_arp_item.dataassinatura.value_counts(1, dropna=False).head()

dataassinatura
2026-03-31T00:00:00    0.019835
2026-03-11T00:00:00    0.015478
2026-03-18T00:00:00    0.014128
2026-04-08T00:00:00    0.013472
2026-03-24T00:00:00    0.013188
Name: proportion, dtype: float64

#### Sem data dominante.

### `datavigenciainicial`
Data inicial da vigência

In [40]:
raw_arp_item.datavigenciainicial.value_counts(1, dropna=False).head()

datavigenciainicial
2026-04-01    0.021763
2026-03-19    0.015970
2026-04-09    0.015681
2026-03-13    0.015509
2026-04-07    0.014727
Name: proportion, dtype: float64

#### Sem data dominante, concentrado em 2026.

### `datavigenciafinal`
Data final da vigência

In [41]:
raw_arp_item.datavigenciafinal.value_counts(0, dropna=False).head()

datavigenciafinal
2027-04-01    7514
2027-04-07    5595
2027-03-19    5143
2027-03-30    4812
2027-04-09    4803
Name: count, dtype: int64

#### Datas concentradas em 2027 — mesmo padrão de vigência de 1 ano visto no cabeçalho da ata.

### `numeroitem`
Número do item

In [42]:
raw_arp_item.numeroitem.value_counts(1, dropna=False).head()

numeroitem
00001    0.030956
00002    0.026104
NaN      0.024504
00003    0.023467
00004    0.021495
Name: proportion, dtype: float64

#### 2,5% dos valores nulos; item nº1 é o mais comum (3,1%).

### `codigoitem`
Código do item no catálogo

In [43]:
raw_arp_item.codigoitem.value_counts(1, dropna=False).head()

codigoitem
19810    0.010170
22225    0.003559
13463    0.003466
3670     0.003260
NaN      0.002139
Name: proportion, dtype: float64

#### Disperso, item mais frequente (19810) concentra 1% das linhas; 0,2% nulos.

### `descricaoitem`
Descrição do item

In [44]:
raw_arp_item.descricaoitem.value_counts(0, dropna=False).head()

descricaoitem
NaN                                                                                                     9404
Manutenção de Equipamentos de Cozinha Industrial                                                        1106
Manutenção / Reforma Predial                                                                             658
MATERIAL LABORATÓRIO, TIPO SUPORTE, MATERIAL METAL, ADICIONAL CIRCULAR, APLICAÇÃO PARA PIPETAGEM         658
PEÇA MECÂNICA/ELÉTRICA - VEÍCULO AUTOMOTIVO, TIPO PARAFUSO, APLICAÇÃO VEÍCULO AUTOMOTIVO, USO 1 RODA     594
Name: count, dtype: int64

#### 2,5% dos valores nulos; manutenção de equipamentos e materiais de laboratório entre as descrições mais frequentes.

### `tipoitem`
Tipo do item (ex.: material ou serviço)

In [45]:
raw_arp_item.tipoitem.value_counts(1, dropna=False).head()

tipoitem
Material    0.879848
Serviço     0.120152
Name: proportion, dtype: float64

#### 88% Material, 12% Serviço.

### `quantidadehomologadaitem`
Quantidade homologada para o item

In [46]:
raw_arp_item.quantidadehomologadaitem.value_counts(1, dropna=False).head()

quantidadehomologadaitem
10.0     0.043125
100.0    0.035868
20.0     0.035271
2.0      0.034963
50.0     0.033895
Name: proportion, dtype: float64

#### Disperso, valores redondos (10, 100, 20, 2, 50) concentram as maiores frequências.

### `classificacaofornecedor`
Classificação do fornecedor (ex.: ME/EPP)

In [47]:
raw_arp_item.classificacaofornecedor.value_counts(1, dropna=False).head()

classificacaofornecedor
1    1.0
Name: proportion, dtype: float64

#### 100% com valor 1 — coluna sem variação, não discrimina nada nesta base.

### `nifornecedor`
Número de identificação do fornecedor

In [48]:
raw_arp_item.nifornecedor.value_counts(1, dropna=False).head()

nifornecedor
NaN               0.011715
12433700000159    0.008229
28820255000110    0.005944
46884097000143    0.005662
44734671002286    0.005211
Name: proportion, dtype: float64

#### 1,2% dos valores nulos; fornecedor mais frequente concentra 0,8% dos itens.

### `nomerazaosocialfornecedor`
Nome ou razão social do fornecedor

In [49]:
raw_arp_item.nomerazaosocialfornecedor.value_counts(0, dropna=False).head()

nomerazaosocialfornecedor
NaN                                                                                4539
NUTRICELLI COMERCIO DE ALIMENTOS LTDA                                              3158
SUPREMA DENTAL IMPORTACAO, EXPORTACAO E COMERCIO DE PRODUTOS ODONTOLOGICOS LTDA    2281
GOLDEN PRODUTOS ODONTOLOGICOS LTDA                                                 2173
CRISTALIA PRODUTOS QUIMICOS FARMACEUTICOS LTDA                                     2000
Name: count, dtype: int64

#### 1,2% nulos (coerente com `nifornecedor`); Nutricelli, Suprema Dental e Golden Produtos Odontológicos entre os mais frequentes.

### `quantidadehomologadavencedor`
Quantidade homologada para o fornecedor vencedor

In [50]:
raw_arp_item.quantidadehomologadavencedor.value_counts(1, dropna=False).head()

quantidadehomologadavencedor
10     0.043125
100    0.035868
20     0.035266
2      0.034961
50     0.033900
Name: proportion, dtype: float64

#### Praticamente idêntica a `quantidadehomologadaitem`.

### `valorunitario`
Valor unitário

In [51]:
raw_arp_item.valorunitario.value_counts(1, dropna=False).head()

valorunitario
1.0     0.005396
15.0    0.003427
10.0    0.003304
30.0    0.003176
20.0    0.003127
Name: proportion, dtype: float64

#### Disperso, sem valor dominante — R$1,00 é o mais frequente (0,5%).

### `valortotal`
Valor total

In [52]:
raw_arp_item.valortotal.value_counts(1, dropna=False).head()

valortotal
3000.0    0.002272
1200.0    0.002160
1800.0    0.002030
6000.0    0.002009
600.0     0.001986
Name: proportion, dtype: float64

#### Disperso, sem valor dominante.

### `maximoadesao`
Quantidade máxima permitida para adesão ao item

In [53]:
raw_arp_item.maximoadesao.value_counts(1, dropna=False).head()

maximoadesao
0.0      0.368805
20.0     0.027829
200.0    0.024108
40.0     0.022482
100.0    0.022078
Name: proportion, dtype: float64

#### 37% dos valores iguais a zero (sem limite de adesão definido).

### `nomeunidadegerenciadora`
Nome da unidade gerenciadora

In [54]:
raw_arp_item.nomeunidadegerenciadora.value_counts(1, dropna=False).head()

nomeunidadegerenciadora
GRUPAMENTO DE APOIO DO GALEAO            0.011830
PREFEITURA MUN. DE FRANCISCO SANTOS      0.007635
HOSPITAL UNIVERSITARIO DE SANTA MARIA    0.006691
HOSPITAL MILITAR DE AREA DE SAO PAULO    0.006509
PREF.MUN.DE BOM JESUS DO ITABAPOANA      0.006420
Name: proportion, dtype: float64

#### Redundante com `codigounidadegerenciadora` — mesmos percentuais.

### `nomemodalidadecompra`
Nome da modalidade de compra

In [55]:
raw_arp_item.nomemodalidadecompra.value_counts(1, dropna=False).head()

nomemodalidadecompra
Pregão             0.991224
Dispensa           0.007379
Inexigibilidade    0.001027
Concorrência       0.000370
Name: proportion, dtype: float64

#### 99% Pregão, mesma distribuição de `codigomodalidadecompra`.

### `idcompra`
Identificador único da compra

In [56]:
raw_arp_item.idcompra.value_counts(1, dropna=False).head()

idcompra
NaN                  0.110308
12064505900172025    0.010681
98108305900092026    0.003747
18302305900202025    0.003265
16009505900142025    0.002019
Name: proportion, dtype: float64

#### 11% dos valores nulos, mesmo padrão de `numerocompra`.

### `numerocontrolepncpcompra`
Número de controle da compra no PNCP

In [57]:
raw_arp_item.numerocontrolepncpcompra.value_counts(1, dropna=False).head()

numerocontrolepncpcompra
00394429000100-1-000524/2025    0.010681
NaN                             0.009388
06553713000169-1-000013/2026    0.003747
00662270000168-1-000164/2025    0.003265
00394452000103-1-025261/2025    0.002014
Name: proportion, dtype: float64

#### 0,9% dos valores nulos.

### `datahorainclusao`
Data e hora de inclusão no sistema

In [58]:
raw_arp_item.datahorainclusao.value_counts(1, dropna=False).head()

datahorainclusao
2026-04-06T12:55:03    0.000646
2026-04-06T00:22:39    0.000573
2026-04-06T12:55:04    0.000542
2026-03-13T16:50:59    0.000511
2026-04-13T09:30:24    0.000464
Name: proportion, dtype: float64

#### Sem timestamp dominante.

### `datahoraatualizacao`
Data e hora da última atualização

In [59]:
raw_arp_item.datahoraatualizacao.value_counts(1, dropna=False).head()

datahoraatualizacao
2026-04-06T12:55:03    0.000646
2026-04-06T00:22:39    0.000573
2026-04-06T12:55:04    0.000542
2026-03-13T16:50:59    0.000511
2026-04-13T09:30:24    0.000464
Name: proportion, dtype: float64

#### Praticamente idêntica a `datahorainclusao` nos valores mais frequentes.

### `quantidadeempenhada`
Quantidade empenhada do item

In [60]:
raw_arp_item.quantidadeempenhada.value_counts(1, dropna=False).head()

quantidadeempenhada
0.0    1.0
Name: proportion, dtype: float64

#### (TO DO) 100% igual a zero — a coluna não parece ser usada nesta tabela; o controle real de empenho está em `raw_arp_empenhos_saldo`.

### `percentualmaiordesconto`
Percentual de maior desconto

In [61]:
raw_arp_item.percentualmaiordesconto.value_counts(1, dropna=False).head()

percentualmaiordesconto
0.00     0.988678
0.01     0.000808
25.00    0.000756
30.00    0.000477
50.00    0.000461
Name: proportion, dtype: float64

#### 99% igual a zero — desconto raramente registrado.

### `situacaosicaf`
Situação do fornecedor no SICAF

In [62]:
raw_arp_item.situacaosicaf.value_counts(1, dropna=False).head()

situacaosicaf
1    0.997074
2    0.002926
Name: proportion, dtype: float64

#### 99,7% com valor 1.

### `datahoraexclusao`
Data e hora de exclusão

In [63]:
raw_arp_item.datahoraexclusao.value_counts(1, dropna=False).head()

datahoraexclusao
NaN                    0.999419
2026-06-11T08:09:25    0.000057
2026-06-11T08:10:13    0.000052
2026-05-29T16:24:38    0.000044
2026-06-11T08:08:18    0.000036
Name: proportion, dtype: float64

#### 99,9% dos valores nulos, coerente com os 0,06% de `itemexcluido = True`.

### `itemexcluido`
Indicador se o item foi excluído (0 – False/Não, 1 – True/Sim)

In [64]:
raw_arp_item.itemexcluido.value_counts(1, dropna=False).head()

itemexcluido
False    0.999419
True     0.000581
Name: proportion, dtype: float64

#### 0,06% dos itens excluídos.

### `numerocontrolepncpata`
Número de controle da ata no PNCP

In [65]:
raw_arp_item.numerocontrolepncpata.value_counts(1, dropna=False).head()

numerocontrolepncpata
00394429000100-1-000524/2025-000001    0.010681
NaN                                    0.009388
06553713000169-1-000013/2026-000001    0.003747
00662270000168-1-000164/2025-000001    0.002749
29115474000160-1-000165/2026-000001    0.001543
Name: proportion, dtype: float64

#### 0,9% dos valores nulos.

### `codigopdm`
Código do PDM (Padrão Descritivo de Material)

In [66]:
raw_arp_item.codigopdm.value_counts(1, dropna=False).head()

codigopdm
NaN        0.122289
19789.0    0.008435
19772.0    0.007755
19774.0    0.007069
30040.0    0.006986
Name: proportion, dtype: float64

#### 12% dos valores nulos.

### `nomepdm`
Nome do PDM

In [67]:
raw_arp_item.nomepdm.value_counts(1, dropna=False).head()

nomepdm
NaN                       0.122289
FRUTA                     0.008753
LEGUME IN NATURA          0.008086
CONDIMENTO                0.007293
FIO DE SUTURA AGULHADO    0.007132
Name: proportion, dtype: float64

#### 12% nulos, coerente com `codigopdm`; Fruta, Legume in natura e Condimento entre os mais frequentes — mesmo padrão visto em `raw_contratos_item`.

## `raw_arp_unidades_item`
Unidades participantes da ata por item — saldo disponível para cada unidade.

| Coluna | Descrição |
|---|---|
| `numeroataregistropreco` **FK** → `raw_arp_item` | Número da ata |
| `codigounidadegerenciadora` **FK** | UASG gerenciadora |
| `numeroitem` **FK** | Item da ata |
| `codigounidade` | Código da unidade participante |
| `nomeunidade` | Nome da unidade |
| `tipounidade` | Gerenciadora ou Participante |
| `quantidaderegistrada` | Quantidade total registrada para a unidade |
| `saldoadesoes` | Saldo disponível para adesões (carona) |
| `saldoremanejamentoempenho` | Saldo disponível para empenho |
| `aceitaadesao` | Se aceita adesão de não participantes |

> **Nota:** Os campos `numeroataregistropreco`, `codigounidadegerenciadora` e `numeroitem` foram adicionados na ingestão a partir dos parâmetros da requisição — a API não os retorna no corpo da resposta.


In [68]:
raw_arp_unidades_item.shape

(33441, 21)

In [69]:
raw_arp_unidades_item.duplicated().sum()

0

In [70]:
raw_arp_unidades_item.head(2)

,numeroata,unidadegerenciadora,numeroitem,codigopdm,descricaoitem,fornecedor,quantidaderegistrada,saldoadesoes,saldoremanejamentoempenho,qtdlimiteadesao,qtdlimiteinformadocompra,aceitaadesao,datahorainclusao,datahoraatualizacao,datahoraexclusao,codigounidade,nomeunidade,tipounidade,dt_ingest,numeroataregistropreco,codigounidadegerenciadora
0,00000/2026,987607,00004,02249,"ÁLCOOL ANIDRO COMBUSTÍVEL, CLASSIFICAÇÃO COMUM, USO PARA AUTOMOTIVOS, COMPOS...",72469687000209 - COMBUSTIVEIS GUAPO LTDA,50000.0,0.0,50000.0,0.0,0.0,False,2026-06-18T11:57:31,2026-06-18T11:57:31,NaN,987607,PREFEITURA MUNICIPAL DE IRATI/PR,GERENCIADORA,2026-06-30T23:20:36.951638,NaN,NaN
1,00001/2026,70020,00006,NaN,Remoção de película para o controle solar na REGIÃO 3. Demais especificações...,21539296000193 - DIEGO PELICULAS PREMIUM LTDA,150.0,300.0,150.0,300.0,300.0,True,2026-05-28T16:17:55,2026-05-28T16:17:56,NaN,070020,TRIBUNAL REGIONAL ELEITORAL DE SANTA CATARINA,GERENCIADORA,2026-06-30T23:20:37.905167,NaN,NaN


### `numeroata`
Número da Ata de Registro de Preços

In [71]:
raw_arp_unidades_item.numeroata.value_counts(1, dropna=False).head()

numeroata
00001/2026    0.779403
00002/2026    0.187285
00002/2025    0.023923
00001/2025    0.009330
00000/2026    0.000060
Name: proportion, dtype: float64

#### 78% concentrados na ata 00001/2026, bem mais concentrado que a distribuição do cabeçalho — sugere que essa ata tem número de unidades participantes muito acima da média.

### `unidadegerenciadora`
Código da unidade gerenciadora

In [72]:
raw_arp_unidades_item.unidadegerenciadora.value_counts(1, dropna=False).head()

unidadegerenciadora
160163    0.032983
183023    0.031518
170045    0.024551
158276    0.023175
160058    0.021142
Name: proportion, dtype: float64

#### Disperso, unidade mais frequente concentra 3,3% das linhas.

### `numeroitem`
Número do item

In [73]:
raw_arp_unidades_item.numeroitem.value_counts(1, dropna=False).head()

numeroitem
00001    0.048623
00002    0.038635
00003    0.031877
00004    0.026853
00005    0.025508
Name: proportion, dtype: float64

#### Item nº1 é o mais comum (4,9%).

### `codigopdm`
Código do PDM

In [74]:
raw_arp_unidades_item.codigopdm.value_counts(1, dropna=False).head()

codigopdm
NaN      0.259113
19774    0.011961
13546    0.011632
05598    0.010616
19772    0.009838
Name: proportion, dtype: float64

#### 26% dos valores nulos — mais que os 12% de `raw_arp_item`.

### `descricaoitem`
Descrição do item

In [75]:
raw_arp_unidades_item.descricaoitem.value_counts(0, dropna=False).head()

descricaoitem
Outros Serviços para a Infraestrutura de Tecnologia da Informação e Comunicaçã o (TIC)                                                                                                                        561
Ar Condicionado - Manutenção de Sistemas / Limpeza                                                                                                                                                            540
CONEXÃO METÁLICA, MATERIAL AÇO GALVANIZADO, BITOLA 1/4 POL, APLICAÇÃO MANGUEIRAS HIDRÁULICAS, CARACTERÍSTICAS ADICIONAIS MODELO 10MM X 1/4 X 2T 10 4 PO, TIPO OLHAL BANJO, PRESSÃO MÁXIMA TRABALHO 900 PSI    355
Instalação e Montagem de Rede Local de Conectividade                                                                                                                                                          321
Manutenção em Equipamentos de Consultório Odontológico                                                                                            

#### Itens de TIC, ar condicionado e conexões metálicas entre os mais frequentes.

### `fornecedor`
Identificação do fornecedor

In [76]:
raw_arp_unidades_item.fornecedor.value_counts(0, dropna=False).head()

fornecedor
55948448000123 - J5 SOLUCOES EMPRESARIAIS LTDA                   1103
54303736000168 - SUNERGY FARM AND CONSTRUCTION LTDA              1054
47531706000143 - ASTA MOBILI MOVEIS LTDA                          821
08368875000152 - FORTLINE INDUSTRIA E COMERCIO DE MOVEIS LTDA     771
09254386000132 - DIGITAL PAPELARIA E INFORMATICA LTDA             696
Name: count, dtype: int64

#### Disperso, fornecedor mais frequente concentra cerca de 3% das linhas.

### `quantidaderegistrada`
Quantidade registrada do item

In [77]:
raw_arp_unidades_item.quantidaderegistrada.value_counts(1, dropna=False).head()

quantidaderegistrada
10.0    0.078048
1.0     0.052989
20.0    0.052510
2.0     0.050298
50.0    0.043061
Name: proportion, dtype: float64

#### Valores redondos (10, 1, 20, 2, 50) concentram as maiores frequências, mesmo padrão de `quantidadehomologadaitem`.

### `saldoadesoes`
Saldo disponível para adesão (carona)

In [78]:
raw_arp_unidades_item.saldoadesoes.value_counts(1, dropna=False).head()

saldoadesoes
0.0     0.371490
20.0    0.021800
4.0     0.020723
2.0     0.016178
8.0     0.015041
Name: proportion, dtype: float64

#### 37% dos valores iguais a zero (sem saldo disponível para adesão).

### `saldoremanejamentoempenho`
Saldo disponível para remanejamento de empenho

In [79]:
raw_arp_unidades_item.saldoremanejamentoempenho.value_counts(1, dropna=False).head()

saldoremanejamentoempenho
10.0    0.077629
1.0     0.053467
20.0    0.052062
2.0     0.050387
50.0    0.042941
Name: proportion, dtype: float64

#### Distribuição parecida com `quantidaderegistrada`.

### `qtdlimiteadesao`
Quantidade limite para adesão

In [80]:
raw_arp_unidades_item.qtdlimiteadesao.value_counts(1, dropna=False).head()

qtdlimiteadesao
0.0     0.371161
20.0    0.026255
4.0     0.021261
40.0    0.017045
2.0     0.016477
Name: proportion, dtype: float64

#### 37% dos valores iguais a zero.

### `qtdlimiteinformadocompra`
Quantidade limite para remanejamento na compra

In [81]:
raw_arp_unidades_item.qtdlimiteinformadocompra.value_counts(1, dropna=False).head()

qtdlimiteinformadocompra
0.0     0.361921
NaN     0.028797
20.0    0.024670
4.0     0.020484
40.0    0.016148
Name: proportion, dtype: float64

#### 36% iguais a zero, 2,9% nulos.

### `aceitaadesao`
Indica se a unidade aceita adesão de não participantes

In [82]:
raw_arp_unidades_item.aceitaadesao.value_counts(1, dropna=False).head()

aceitaadesao
True     0.627942
False    0.371161
NaN      0.000897
Name: proportion, dtype: float64

#### 63% aceitam adesão, 37% não; 0,09% nulos.

### `datahorainclusao`
Data e hora da inclusão

In [83]:
raw_arp_unidades_item.datahorainclusao.value_counts(1, dropna=False).head()

datahorainclusao
2026-05-12T14:38:50    0.000419
2025-05-30T11:34:27    0.000359
2026-05-06T10:58:43    0.000269
2026-03-18T14:55:19    0.000239
2026-03-19T18:38:27    0.000239
Name: proportion, dtype: float64

#### Sem timestamp dominante.

### `datahoraatualizacao`
Data e hora da última atualização

In [84]:
raw_arp_unidades_item.datahoraatualizacao.value_counts(1, dropna=False).head()

datahoraatualizacao
2026-06-10T15:35:32    0.002602
2026-02-03T12:48:40    0.002362
2026-02-03T12:48:41    0.001824
2026-03-06T11:37:00    0.001645
2026-03-18T15:06:33    0.001585
Name: proportion, dtype: float64

#### Leve concentração em datas de fevereiro/março de 2026.

### `datahoraexclusao`
Data e hora da exclusão (se houver)

In [85]:
raw_arp_unidades_item.datahoraexclusao.value_counts(1, dropna=False).head()

datahoraexclusao
NaN    1.0
Name: proportion, dtype: float64

#### 100% dos valores nulos — nenhuma linha excluída na base.

### `codigounidade`
Código da unidade participante

In [86]:
raw_arp_unidades_item.codigounidade.value_counts(1, dropna=False).head()

codigounidade
183023    0.031518
791580    0.015909
194049    0.005891
927155    0.005801
158368    0.005353
Name: proportion, dtype: float64

#### Disperso, unidade mais frequente concentra 3,2% das linhas.

### `nomeunidade`
Nome da unidade

In [87]:
raw_arp_unidades_item.nomeunidade.value_counts(1, dropna=False).head()

nomeunidade
INSTITUTO NAC.DE METROLOG. QUALID. E TECNOLOG    0.031518
BASE ALMIRANTE CASTRO E SILVA                    0.015909
FUNAI-COORDENACAO REGIONAL DO JURUA/AC           0.005891
DIRETORIA DE SAÚDE DA POLÍCIA MILITAR DO ES      0.005801
INST.FED.DO RN/CAMPUS NATAL - ZONA NORTE         0.005353
Name: proportion, dtype: float64

#### Redundante com `codigounidade` — mesmos percentuais.

### `tipounidade`
Tipo da unidade (Gerenciadora ou Participante)

In [88]:
raw_arp_unidades_item.tipounidade.value_counts(1, dropna=False).head()

tipounidade
GERENCIADORA    0.552525
PARTICIPANTE    0.447475
Name: proportion, dtype: float64

#### 55% Gerenciadora, 45% Participante.

### `numeroataregistropreco`
Número da ata (campo replicado a partir dos parâmetros da requisição na ingestão)

In [89]:
raw_arp_unidades_item.numeroataregistropreco.value_counts(1, dropna=False).head()

numeroataregistropreco
00001/2026    0.768398
00002/2026    0.187285
00002/2025    0.023923
NaN           0.011752
00001/2025    0.008612
Name: proportion, dtype: float64

#### 1,2% dos valores nulos — diferente do equivalente `numeroata`, que não tem nulos, apesar de ambos se referirem ao número da ata.

### `codigounidadegerenciadora`
Código da unidade gerenciadora (campo replicado a partir dos parâmetros da requisição na ingestão)

In [90]:
raw_arp_unidades_item.codigounidadegerenciadora.value_counts(1, dropna=False).head()

codigounidadegerenciadora
160163    0.032983
183023    0.031518
170045    0.024551
158276    0.023175
160058    0.021142
Name: proportion, dtype: float64

#### Idêntica a `unidadegerenciadora` — mesmo campo duplicado, injetado na ingestão a partir dos parâmetros da requisição.

## `raw_arp_adesoes_item`
Adesões ("caronas") aprovadas por item da ata.

| Coluna | Descrição |
|---|---|
| `numeroataregistropreco` **FK** → `raw_arp_item` | Número da ata |
| `codigounidadegerenciadora` **FK** | UASG gerenciadora |
| `numeroitem` **FK** | Item da ata |
| `unidadenaoparticipante` | Código da unidade que aderiu (carona) |
| `quantidadeaprovadaadesao` | Quantidade aprovada na adesão |
| `dataaprovacaoanalise` | Data de aprovação da adesão |

> **Nota:** Os campos de contexto da ata foram injetados na ingestão — a API não os retorna.


In [91]:
raw_arp_adesoes_item.shape

(3389, 9)

In [92]:
raw_arp_adesoes_item.duplicated().sum()

9

In [93]:
raw_arp_adesoes_item.head(3)

,numeroataregistropreco,codigounidadegerenciadora,numeroitem,numeroata,unidadegerenciadora,unidadenaoparticipante,dataaprovacaoanalise,quantidadeaprovadaadesao,dt_ingest
0,00001/2026,090030,00007,00001/2026,90030,090026 - SECRETARIA DO CONSELHO DA JUSTICA FEDERAL,2026-05-29T14:55:39,80.0,2026-06-30T23:35:54.657321
1,00001/2026,090030,00008,00001/2026,90030,090026 - SECRETARIA DO CONSELHO DA JUSTICA FEDERAL,2026-05-29T14:55:39,10.0,2026-06-30T23:36:04.173838
2,00001/2026,090030,00012,00001/2026,90030,090026 - SECRETARIA DO CONSELHO DA JUSTICA FEDERAL,2026-05-29T14:55:39,160.0,2026-06-30T23:36:06.729189


### `numeroataregistropreco`
Número da ata (campo replicado a partir dos parâmetros da requisição na ingestão)

In [94]:
raw_arp_adesoes_item.numeroataregistropreco.value_counts(1, dropna=False).head()

numeroataregistropreco
00001/2026    0.936264
00002/2026    0.058719
00001/2025    0.002656
00002/2025    0.002361
Name: proportion, dtype: float64

#### 94% concentrados na ata 00001/2026 — concentração ainda maior que em `raw_arp_unidades_item`.

### `codigounidadegerenciadora`
Código da unidade gerenciadora (campo replicado a partir dos parâmetros da requisição na ingestão)

In [95]:
raw_arp_adesoes_item.codigounidadegerenciadora.value_counts(1, dropna=False).head()

codigounidadegerenciadora
183023    0.311891
389133    0.140159
170045    0.093833
160478    0.043081
158368    0.037179
Name: proportion, dtype: float64

#### 31% concentrados em uma única UASG (183023), bem mais concentrado que nas outras tabelas de ARP.

### `numeroitem`
Número do item na ata (campo replicado a partir dos parâmetros da requisição na ingestão)

In [96]:
raw_arp_adesoes_item.numeroitem.value_counts(1, dropna=False).head()

numeroitem
00001    0.063145
00004    0.024786
00002    0.022425
00003    0.021835
00005    0.020950
Name: proportion, dtype: float64

#### Item nº1 é o mais comum (6,3%).

### `numeroata`
Número da Ata de Registro de Preços

In [97]:
raw_arp_adesoes_item.numeroata.value_counts(1, dropna=False).head()

numeroata
00001/2026    0.936264
00002/2026    0.058719
00001/2025    0.002656
00002/2025    0.002361
Name: proportion, dtype: float64

#### Idêntica a `numeroataregistropreco` — mesmo campo duplicado, injetado na ingestão.

### `unidadegerenciadora`
Código da unidade gerenciadora

In [98]:
raw_arp_adesoes_item.unidadegerenciadora.value_counts(1, dropna=False).head()

unidadegerenciadora
183023    0.311891
389133    0.140159
170045    0.093833
160478    0.043081
158368    0.037179
Name: proportion, dtype: float64

#### Idêntica a `codigounidadegerenciadora` — mesmo campo duplicado, injetado na ingestão.

### `unidadenaoparticipante`
Código da unidade não participante (órgão carona)

In [99]:
raw_arp_adesoes_item.unidadenaoparticipante.value_counts(1, dropna=False).head()

unidadenaoparticipante
393016 - SUPERINTENDENCIA REGIONAL NO ESTADO PA - DNIT    0.293007
926655 - CONSELHO FEDERAL DE ODONTOLOGIA                  0.043376
927581 - CONSELHO REGIONAL DE ADMINISTRAÇÃO DO AM         0.042490
927457 - CONSELHO REGIONAL DOS TÉCNICOS INDUSTRIAIS       0.036294
795500 - BASE DE FUZILEIROS NAVAIS DO RIO MERITI          0.018885
Name: proportion, dtype: float64

#### 29% concentrados em uma única unidade (DNIT-PA) — forte concentração de caronas em poucos órgãos.

### `dataaprovacaoanalise`
Data da aprovação da análise da adesão

In [100]:
raw_arp_adesoes_item.dataaprovacaoanalise.value_counts(1, dropna=False).head()

dataaprovacaoanalise
2026-03-05T09:28:28    0.071112
2026-03-05T09:45:57    0.059014
2026-03-05T09:34:18    0.056654
2026-03-05T09:17:46    0.054293
2026-03-05T09:39:59    0.051933
Name: proportion, dtype: float64

#### Datas concentradas em 5 de março de 2026 — sugere aprovação em lote.

### `quantidadeaprovadaadesao`
Quantidade aprovada para a adesão

In [101]:
raw_arp_adesoes_item.quantidadeaprovadaadesao.value_counts(1, dropna=False).head()

quantidadeaprovadaadesao
NaN     0.100325
1.0     0.078784
5.0     0.076719
10.0    0.066981
2.0     0.053113
Name: proportion, dtype: float64

#### 10% dos valores nulos.

## `raw_arp_empenhos_saldo`
Saldo de empenhos por ata e unidade gerenciadora.

| Coluna | Descrição |
|---|---|
| `numeroataregistropreco` **FK** → `raw_arp` | Número da ata |
| `codigounidadegerenciadora` **FK** | UASG gerenciadora |
| `tipo` | Tipo de empenho |
| `quantidaderegistrada` | Quantidade total registrada |
| `quantidadeempenhada` | Quantidade já empenhada |
| `saldoempenho` | Saldo disponível para empenho |

> **Nota:** Os campos de contexto foram injetados na ingestão. A tabela tem apenas 298 registros pois o endpoint de empenhos é extremamente lento (~25s/requisição) e a ingestão está incompleta.


In [102]:
raw_arp_empenhos_saldo.shape

(298, 11)

In [103]:
raw_arp_empenhos_saldo.duplicated().sum()

0

In [104]:
raw_arp_empenhos_saldo.head(3)

,numeroataregistropreco,codigounidadegerenciadora,numeroitem,unidade,tipo,quantidaderegistrada,quantidadeempenhada,saldoempenho,datahorainclusao,datahoraatualizacao,dt_ingest
0,00001/2025,153149,00004,153148 - INSTITUTO DE NEUROL. DEOLINDO COUTO DA UFRJ,PARTICIPANTE,300.0,300.0,0.0,NaN,2026-02-11T10:47:55,2026-06-30T23:35:21.981155
1,00001/2026,110404,00006,110404 - DEPARTAMENTO DE ADMINISTRACAO INTERNA-MD,GERENCIADORA,1.0,1.0,0.0,NaN,2026-01-22T17:34:30,2026-06-30T23:46:54.271867
2,00001/2026,110404,00018,110404 - DEPARTAMENTO DE ADMINISTRACAO INTERNA-MD,GERENCIADORA,10.0,10.0,0.0,NaN,2026-01-22T17:34:30,2026-06-30T23:46:54.271867


### `numeroataregistropreco`
Número da ata (campo replicado a partir dos parâmetros da requisição na ingestão)

In [105]:
raw_arp_empenhos_saldo.numeroataregistropreco.value_counts(1, dropna=False).head()

numeroataregistropreco
00001/2026    0.523490
00003/2026    0.211409
00002/2026    0.177852
00002/2025    0.077181
00003/2025    0.006711
Name: proportion, dtype: float64

#### 52% concentrados na ata 00001/2026, refletindo a base pequena (298 linhas) e incompleta descrita na nota da tabela.

### `codigounidadegerenciadora`
Código da unidade gerenciadora (campo replicado a partir dos parâmetros da requisição na ingestão)

In [106]:
raw_arp_empenhos_saldo.codigounidadegerenciadora.value_counts(1, dropna=False).head()

codigounidadegerenciadora
153978    0.147651
158148    0.087248
160395    0.087248
153289    0.073826
160222    0.063758
Name: proportion, dtype: float64

#### 15% concentrados em uma única UASG — base pequena, resultado sensível a poucos registros.

### `numeroitem`
Número do item

In [107]:
raw_arp_empenhos_saldo.numeroitem.value_counts(1, dropna=False).head()

numeroitem
00002    0.067114
00004    0.053691
00001    0.046980
00005    0.046980
00006    0.043624
Name: proportion, dtype: float64

#### Item nº2 é o mais comum (6,7%).

### `unidade`
Unidade relacionada ao empenho

In [108]:
raw_arp_empenhos_saldo.unidade.value_counts(1, dropna=False).head()

unidade
153978 - INST.NACIONAL DE EST.E PESQUISAS EDUCACIONAIS    0.147651
158148 - INST. FED. DE EDUC.,CIENC. E TEC. DE RONDONIA    0.087248
160399 - HOSPITAL MILITAR DE AREA DE PORTO ALEGRE         0.087248
153289 - FACULDADE DE MEDICINA/UFMG                       0.073826
160222 - 5. BATALHAO DE SUPRIMENTO                        0.063758
Name: proportion, dtype: float64

#### Redundante com `codigounidadegerenciadora` — mesmos percentuais.

### `tipo`
Tipo de registro (ex.: empenho, saldo)

In [109]:
raw_arp_empenhos_saldo.tipo.value_counts(1, dropna=False).head()

tipo
GERENCIADORA        0.862416
NÃO PARTICIPANTE    0.100671
PARTICIPANTE        0.036913
Name: proportion, dtype: float64

#### 86% Gerenciadora, 10% Não Participante, 3,7% Participante.

### `quantidaderegistrada`
Quantidade total registrada

In [110]:
raw_arp_empenhos_saldo.quantidaderegistrada.value_counts(1, dropna=False).head()

quantidaderegistrada
10.0    0.073826
3.0     0.067114
20.0    0.057047
1.0     0.046980
12.0    0.040268
Name: proportion, dtype: float64

#### Valores redondos concentram as maiores frequências, mesmo padrão de `raw_arp_unidades_item`.

### `quantidadeempenhada`
Quantidade que já foi empenhada

In [111]:
raw_arp_empenhos_saldo.quantidadeempenhada.value_counts(1, dropna=False).head()

quantidadeempenhada
1.0     0.090604
2.0     0.077181
0.0     0.077181
20.0    0.060403
3.0     0.053691
Name: proportion, dtype: float64

#### Bem mais variado que a mesma coluna em `raw_arp_item`, que é 100% zero — aqui o campo é de fato utilizado.

### `saldoempenho`
Saldo restante para empenho

In [112]:
raw_arp_empenhos_saldo.saldoempenho.value_counts(1, dropna=False).head()

saldoempenho
0.0     0.308725
8.0     0.040268
6.0     0.036913
30.0    0.023490
4.0     0.023490
Name: proportion, dtype: float64

#### 31% dos valores iguais a zero.

### `datahorainclusao`
Data e hora da inclusão do registro

In [113]:
raw_arp_empenhos_saldo.datahorainclusao.value_counts(1, dropna=False).head()

datahorainclusao
NaN    1.0
Name: proportion, dtype: float64

#### 100% dos valores nulos.

### `datahoraatualizacao`
Data e hora da última atualização

In [114]:
raw_arp_empenhos_saldo.datahoraatualizacao.value_counts(1, dropna=False).head()

datahoraatualizacao
2026-02-03T12:58:51    0.144295
2026-02-04T14:34:10    0.083893
2026-02-10T16:24:50    0.063758
2026-02-11T15:22:18    0.033557
2026-02-05T16:19:35    0.030201
Name: proportion, dtype: float64

#### 14% concentrados numa única data (2026-02-03) — base pequena, resultado sensível a poucos registros.

In [115]:
arp = pd.merge(raw_arp, raw_arp_item, on='numeroataregistropreco', how='outer',
                        indicator='_arp_item', suffixes=('_arp', '_arp_item'))

In [116]:
arp._arp_item.value_counts()

_arp_item
both          130563488
right_only          406
left_only            84
Name: count, dtype: int64